This file is to experiment with setting thresholds for our data with NLP-produced similarity scores
- for now, just using some sample file(s) from get_scores()
- sectioning off data by each ..05 of similarity scores
- then, get cohen's kappas on that for comparison

In [1]:
import pandas as pd
from NLP_Eval_for_DE import scores, data
import ast
from sklearn.metrics import cohen_kappa_score, f1_score, confusion_matrix


Welcome to NLP eval for DE, version 1.0.0


In [ ]:
testable_data = data.get_testable_data("Example\\inputs\\case study 2 input-open codes\\unclean version\\hackathon numerical GTs.csv")
codes = data.get_codes("Example\\inputs\\case study 2 input-open codes\\hackathon open codes.csv")
all_scores = scores.get_BART_scores(testable_data, codes)[1]

#split the lots of scores in the "Similarity scores" column into separate columns
all_scores_expanded = all_scores.copy()
#make this go up to 88
all_scores_expanded[[str(i) for i in range(1, 89)]] = pd.DataFrame(all_scores_expanded["Similarity scores"].tolist(), index=all_scores_expanded.index)
# add the "Open code(s)" column of testable_data to all_scores_expanded
all_scores_expanded["Open code(s)"] = testable_data["Open code(s)"].tolist()
#delete the "Similarity scores" column
all_scores_expanded = all_scores_expanded.drop(columns=["Similarity scores"])
all_scores_expanded

Device set to use cuda:0


In [107]:
def eval_filtered(all_scores_expanded, min, max):
    # now filter based on threshold 
    # drop any rows where the highest score from the 10 scores is not between min and max
    all_scores_expanded_filtered = all_scores_expanded[
        (all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) >= min) 
        & (all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) <= max)]

    # now, get cohens kappa score for the filtered data
    ground_truths = all_scores_expanded_filtered["Open code(s)"].tolist()
    predictions = all_scores_expanded_filtered[[str(i) for i in range(1, 89)]].idxmax(axis=1).tolist()
    # convert predictions from strings to ints
    predictions = [int(x) for x in predictions]
    #make these labels also go to 88 instead of 11
    f1s = f1_score(ground_truths, predictions, labels=list(range(1, 89)), average=None, zero_division=0.0) 
    #mtx = confusion_matrix(ground_truths, predictions, labels=list(range(1, 89)))
    kappa = cohen_kappa_score(ground_truths, predictions, weights=None, sample_weight=None)
    # get average f1 (will see later if this is ok)
    f1 = sum(f1s) / len(f1s)
    # count rows below the minimum threshold
    rows_below_min = len(all_scores_expanded[all_scores_expanded[[str(i) for i in range(1, 89)]].max(axis=1) < min])
    return [f1, kappa, rows_below_min]

In [ ]:
rows = []
total_rows = len(all_scores_expanded)
for i, j in [(0.95, 1.0), (0.9, 0.95), (0.85, 0.9), (0.8, 0.85), (0.75, 0.8), (0.7, 0.75), (0.65, 0.7), (0.6, 0.65), (0.55, 0.6), (0.5, 0.55), (0.45, 0.5), (0.4, 0.45), (0.35, 0.4), (0.3, 0.35), (0.25, 0.3), (0.2, 0.25), (0.15, 0.2), (0.1, 0.15), (0.05, 0.1), (0.0, 0.05)]:
    results = eval_filtered(all_scores_expanded, i, j)
    rows_below_min = results[2]
    percentage_below_min = (rows_below_min / total_rows) * 100
    rows.append({"min": i, "max": j, "kappa": results[1], "f1": results[0], "percentage_below_min": percentage_below_min, "rows_below_min": rows_below_min})
thresholded_kappas = pd.DataFrame(rows)
thresholded_kappas.to_csv("thresholding_results\\thrKappas_BART_open_unclean_hackathon.csv", index=False)
thresholded_kappas

C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
C:\Users\ass3352\AppData\Roaming\Python\Python312\site-packages\sklearn\metrics\_classification.py:758: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)


,min,max,kappa,f1,percentage_below_min,rows_below_min
0,0.95,1.00,NaN,0.000000,100.000000,906
1,0.90,0.95,NaN,0.000000,100.000000,906
2,0.85,0.90,NaN,0.000000,100.000000,906
3,0.80,0.85,NaN,0.000000,100.000000,906
4,0.75,0.80,0.333333,0.011364,99.779249,904
5,0.70,0.75,0.000000,0.000000,99.668874,903
6,0.65,0.70,0.000000,0.000000,99.558499,902
7,0.60,0.65,0.000000,0.000000,99.227373,899
8,0.55,0.60,0.000000,0.000000,99.116998,898
9,0.50,0.55,0.133333,0.011364,97.682119,885
